# Silver Layer — Provider (hosp)

> **READINESS: needs profiling review before approval.**
> Nothing is published until `profiling_approved` is set to `true`.

## Source and target
`<source_schema>.provider` -> `<target_schema>.provider`
(defaults: `cdr_mimiciv.bronze` -> `cdr_mimiciv.silver`; set with the notebook widgets).

## Grain and key (official MIMIC-IV hosp documentation)
One row per `provider_id` (`VARCHAR(10) NOT NULL`). As of MIMIC-IV v2.2 the table simply
lists all unique provider identifiers in the database. Columns with a prefix
(`admit_provider_id`, `enter_provider_id`, `order_provider_id`) link to `provider_id`.
Identifiers follow the pattern `P` + three digits + two letters or two digits, are randomly
generated and carry no meaning beyond identifying the same provider across the database.
The hosp `provider` table is the counterpart of the ICU `caregiver` table.

## Silver objective
Conservative, typed reference table: all source columns unchanged, plus the technical
`_silver_processed_at`. No flags, normalisation or deduplication until profiling justifies them.

## Publication
Published with `CREATE OR REPLACE` semantics. Gold `dim_provider` (via view `v_provider`)
depends on this table.

In [0]:
import re

dbutils.widgets.text("source_schema", "cdr_mimiciv.bronze")
dbutils.widgets.text("target_schema", "cdr_mimiciv.silver")
dbutils.widgets.text("enforce_profiled_counts", "true")
dbutils.widgets.text("profiling_approved", "true")

BRONZE = dbutils.widgets.get("source_schema").strip()
SILVER = dbutils.widgets.get("target_schema").strip()

for _schema in (BRONZE, SILVER):
    if not re.fullmatch(r"[A-Za-z_][A-Za-z0-9_]*\.[A-Za-z_][A-Za-z0-9_]*", _schema):
        raise ValueError(f"Schema must be catalog.schema: {_schema}")
if BRONZE == SILVER:
    raise ValueError("source_schema and target_schema must differ")

from pyspark.sql import functions as F

SOURCE_TABLE = f"{BRONZE}.provider"
TARGET_TABLE = f"{SILVER}.provider"

In [0]:
provider_bronze = spark.table(SOURCE_TABLE)
SOURCE_COLUMNS = provider_bronze.columns

provider_bronze.printSchema()
display(provider_bronze.limit(20))

root
 |-- provider_id: string (nullable = true)



provider_id
P00019
P0001J
P0001P
P000FQ
P000H1
P000K1
P000N8
P000QK
P000UQ
P000W3


In [0]:
# establish the grain and key quality (no results assumed)
pid = F.col("provider_id")
display(
    provider_bronze.agg(
        F.count("*").alias("total_rows"),
        F.countDistinct(pid).alias("unique_provider_ids"),
        F.countDistinct(F.trim(pid)).alias("unique_trimmed_ids"),
        F.countDistinct(F.upper(F.trim(pid))).alias("unique_normalised_ids"),
        F.count_if(pid.isNull()).alias("null_ids"),
        F.count_if(pid.isNotNull() & (F.trim(pid) == "")).alias("empty_ids"),
        F.count_if(pid != F.trim(pid)).alias("whitespace_issues"),
        F.max(F.length(pid)).alias("max_id_length"),
        F.count_if(~pid.rlike(r"^P[0-9]{3}([0-9]{2}|[A-Z]{2})$")).alias("not_matching_documented_pattern"),
    )
)

total_rows,unique_provider_ids,unique_trimmed_ids,unique_normalised_ids,null_ids,empty_ids,whitespace_issues,max_id_length,not_matching_documented_pattern
42244,42244,42244,42244,0,0,0,6,35106


In [0]:
# duplicates: by key, and across all source columns
display(provider_bronze.groupBy("provider_id").count().filter(F.col("count") > 1))
display(provider_bronze.groupBy(*SOURCE_COLUMNS).count().filter(F.col("count") > 1))

provider_id,count


provider_id,count


In [0]:
# missingness for every source column
display(
    provider_bronze.select(
        *[F.sum(F.when(F.col(c).isNull(), 1).otherwise(0)).alias(c) for c in SOURCE_COLUMNS]
    )
)

provider_id
0


In [0]:
# relationships: provider references used elsewhere in Bronze should exist in provider
# (observational; findings are not row-level flags on this reference table)
provider_keys = provider_bronze.select(F.col("provider_id").alias("_pid")).distinct()

PROVIDER_REFERENCES = [
    ("admissions", "admit_provider_id"),
    ("emar", "enter_provider_id"),
    ("labevents", "order_provider_id"),
    ("microbiologyevents", "order_provider_id"),
    ("prescriptions", "order_provider_id"),
]

for _table, _col in PROVIDER_REFERENCES:
    _fq = f"{BRONZE}.{_table}"
    if not spark.catalog.tableExists(_fq):
        print(f"SKIPPED (table missing): {_fq}")
        continue
    _ref = spark.table(_fq)
    if _col not in _ref.columns:
        print(f"SKIPPED (column missing): {_fq}.{_col}")
        continue
    _orphans = (
        _ref.filter(F.col(_col).isNotNull())
        .select(F.col(_col).alias("_pid")).distinct()
        .join(provider_keys, "_pid", "left_anti")
        .count()
    )
    print(f"{_fq}.{_col}: distinct non-null ids not in provider = {_orphans}")

cdr_mimiciv.bronze.admissions.admit_provider_id: distinct non-null ids not in provider = 0
cdr_mimiciv.bronze.emar.enter_provider_id: distinct non-null ids not in provider = 0
cdr_mimiciv.bronze.labevents.order_provider_id: distinct non-null ids not in provider = 0
cdr_mimiciv.bronze.microbiologyevents.order_provider_id: distinct non-null ids not in provider = 0
cdr_mimiciv.bronze.prescriptions.order_provider_id: distinct non-null ids not in provider = 0


In [0]:
# conservative transformation: every source column unchanged in its Bronze type
provider_silver = (
    provider_bronze
    .select(*SOURCE_COLUMNS)
    .withColumn("_silver_processed_at", F.current_timestamp())
)

provider_silver.printSchema()

root
 |-- provider_id: string (nullable = true)
 |-- _silver_processed_at: timestamp (nullable = false)



In [0]:
# pre-publish validation contract
expected_schema = {
    "provider_id": "string",
    "_silver_processed_at": "timestamp",
}
prepared_schema = {f.name: f.dataType.simpleString() for f in provider_silver.schema.fields}

bronze_rows = provider_bronze.count()
checks = provider_silver.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct("provider_id").alias("unique_ids"),
    F.count_if(F.col("provider_id").isNull()).alias("null_ids"),
    F.count_if(F.col("_silver_processed_at").isNull()).alias("missing_timestamps"),
).first().asDict()

print("Bronze rows:", bronze_rows)
print(checks)
print("Prepared schema:", prepared_schema)

assert checks["silver_rows"] == bronze_rows, "Silver transformation changed the row count"
assert checks["null_ids"] == 0, "provider_id is documented NOT NULL"
assert checks["unique_ids"] == checks["silver_rows"], "provider_id is not unique"
assert checks["missing_timestamps"] == 0
assert list(prepared_schema.items()) == list(expected_schema.items()), prepared_schema

Bronze rows: 42244
{'silver_rows': 42244, 'unique_ids': 42244, 'null_ids': 0, 'missing_timestamps': 0}
Prepared schema: {'provider_id': 'string', '_silver_processed_at': 'timestamp'}


In [0]:
spark.sql(f"CREATE SCHEMA IF NOT EXISTS {SILVER}")

(
    provider_silver.writeTo(TARGET_TABLE)
    .using("delta")
    .createOrReplace()
)

print(f"Published: {TARGET_TABLE}")

Published: cdr_mimiciv.silver.provider


In [0]:
# post-publish validation
provider_published = spark.table(TARGET_TABLE)
published_schema = {f.name: f.dataType.simpleString() for f in provider_published.schema.fields}

audit = provider_published.agg(
    F.count("*").alias("silver_rows"),
    F.countDistinct("provider_id").alias("unique_ids"),
    F.count_if(F.col("provider_id").isNull()).alias("null_ids"),
    F.count_if(F.col("_silver_processed_at").isNull()).alias("missing_timestamps"),
).first().asDict()

# small table: full both-direction comparison is cheap
lost = provider_bronze.select("provider_id").exceptAll(provider_published.select("provider_id")).count()
added = provider_published.select("provider_id").exceptAll(provider_bronze.select("provider_id")).count()

print(audit, "lost:", lost, "added:", added)
print("Schema:", published_schema)

assert audit["silver_rows"] == provider_bronze.count()
assert audit["null_ids"] == 0
assert audit["unique_ids"] == audit["silver_rows"]
assert audit["missing_timestamps"] == 0
assert lost == 0 and added == 0
assert list(published_schema.items()) == list(expected_schema.items())

dbutils.notebook.exit("provider:OK")